# Part 09. 프로젝트 루트 확인과 4개 CSV 불러오기

## 35. 프로젝트 루트 설정

In [74]:

from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":

    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)

print("데이터 폴더:", data_dir)

print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\2team_shopingmall_dash_board
데이터 폴더: c:\dev\2team_shopingmall_dash_board\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [75]:
import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")

products = pd.read_csv(data_dir / "products.csv")

orders = pd.read_csv(data_dir / "orders.csv")

order_items = pd.read_csv(data_dir / "order_items.csv")

## 37. 기본 구조와 주요 키 확인

In [76]:
datasets = {

    "customers": customers,

    "products": products,

    "orders": orders,

    "order_items": order_items,

}

for name, df in datasets.items():

    print(name, df.shape, df.columns.tolist())

customers (1200, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (300, 4) ['product_id', 'product_name', 'category', 'price']
orders (6000, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (14603, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [77]:

key_checks = {

    "customers.customer_id": customers["customer_id"],

    "products.product_id": products["product_id"],

    "orders.order_id": orders["order_id"],

    "order_items.order_item_id": order_items["order_item_id"],

}

for name, series in key_checks.items():

    print(

        name,

        "결측:", series.isna().sum(),

        "중복:", series.duplicated().sum(),

    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


# Part 10. 컬럼 선택·조건 필터링·정렬

 

## 38. Series와 DataFrame 선택

In [78]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]

print(type(city_series))
print(type(customer_view))
display(customer_view.head())


<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,남성,24,고양
1,2,여성,27,고양
2,3,여성,45,서울
3,4,여성,52,서울
4,5,여성,64,서울


## 39. 단일 조건 필터링

In [79]:

customers_over_30 = customers[

    customers["age"] >= 30
]
print(len(customers), len(customers_over_30))
display(customers_over_30.head())

1200 869


,customer_id,name,gender,age,city,signup_date
2,3,권재윤,여성,45,서울,2024-08-12
3,4,송현진,여성,52,서울,2024-11-18
4,5,김준수,여성,64,서울,2024-06-24
5,6,송은아,여성,35,울산,2024-07-30
6,7,황윤희,남성,40,부산,2026-04-27


# 40. 복합 조건 필터링

 

30세 이상이면서 서울 거주:

In [80]:

seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
display(seoul_over_30.head())


,customer_id,name,gender,age,city,signup_date
2,3,권재윤,여성,45,서울,2024-08-12
3,4,송현진,여성,52,서울,2024-11-18
4,5,김준수,여성,64,서울,2024-06-24
8,9,권재진,남성,54,서울,2024-11-11
9,10,윤예진,남성,46,서울,2025-04-28


In [81]:
# 서울 또는 부산 :

seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)

city
서울    327
부산    114
Name: count, dtype: int64

In [82]:
# 완료 주문이 아닌 주문:

not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)

order_status
배송완료    4862
취소       458
환불       352
배송중      163
배송준비      92
결제완료      73
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [83]:

expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
281,282,베이직 미니 가습기 베이지 P282,생활가전,371000
51,52,컴팩트 공기청정기 베이지 P052,생활가전,364000
191,192,플러스 미니 가습기 화이트 P192,생활가전,362000
271,272,베이직 전기포트 화이트 P272,생활가전,338000
171,172,프리미엄 미니 가습기 블루 P172,생활가전,328000
211,212,라이트 선풍기 그레이 P212,생활가전,307000
251,252,에코 핸디 청소기 블루 P252,생활가전,297000
291,292,컴팩트 미니 가습기 그린 P292,생활가전,294000
101,102,컴팩트 공기청정기 블루 P102,생활가전,283000
61,62,플러스 핸디 청소기 화이트 P062,생활가전,273000


# Part 11. line_total 생성과 전체 주문 금액 구분

 

## 42. 작업용 복사본과 파생 컬럼

In [84]:
order_items_work = order_items.copy()
#데이터 프레임에 새로운 컬럼을 추가할 때는 기존 데이터 프레임을 수정
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

In [85]:

display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()
)


,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,216,4,13600,54400
1,2,1,34,1,87000,87000
2,3,1,187,1,93000,93000
3,4,1,112,1,225000,225000
4,5,2,214,1,64000,64000


In [86]:
print(order_items_work.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         216         4       13600       54400
1              2         1          34         1       87000       87000
2              3         1         187         1       93000       93000
3              4         1         112         1      225000      225000
4              5         2         214         1       64000       64000


In [87]:
# 43. 수작업 검증

sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 54400
파생 컬럼: 54400
일치: True


In [88]:
# 44. 전체 주문상세 금액

 

all_order_amount = order_items_work["line_total"].sum()

print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 1837774300


In [89]:
#45. 병합용 주문 컬럼 선택

orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())

(6000, 4)
   order_id  customer_id  order_date order_status
0         1         1121  2026-05-18          배송중
1         2          993  2026-02-01           환불
2         3          261  2025-05-08         배송완료
3         4          783  2026-02-18         결제완료
4         5          292  2025-06-20         배송완료


In [90]:
#46. 주문상세와 주문 병합

order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)

In [91]:
#47. 병합 검증

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 14603
병합 후 행 수: 14603


order_match
both          14603
left_only         0
right_only        0
Name: count, dtype: int64

In [92]:
#미매칭 확인:

unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match


In [93]:
#48. 완료 주문 분석셋

display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

order_status
배송완료    11764
취소       1128
환불        878
배송중       410
배송준비      232
결제완료      191
Name: count, dtype: int64

In [94]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [95]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)
print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)
print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)

완료 주문상세 행: 0
완료 주문 수: 0
완료 주문 고객 수: 0
완료 주문 매출: 0


In [96]:
#49. 필요한 상품 정보만 선택

products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()

print(products_for_merge.head())

   product_id         product_name category
0           1     프리미엄 웹캠 그레이 P001     전자기기
1           2   스마트 미니 가습기 블랙 P002     생활가전
2           3  컴팩트 후드 티셔츠 그레이 P003       패션
3           4       프로 세럼 화이트 P004       뷰티
4           5  스마트 드립백 커피 화이트 P005       식품


In [97]:
#50. 완료 주문상세와 상품 병합

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [98]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

0 0


product_match
left_only     0
right_only    0
both          0
Name: count, dtype: int64

In [99]:
#51. 카테고리별 매출

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count


In [100]:
tmp = order_items.merge(products, on="product_id", how="left", indicator=True)
print(tmp["_merge"].value_counts())

_merge
both          14603
left_only         0
right_only        0
Name: count, dtype: int64


In [101]:
print("행 수:", len(completed_items))
print("category 결측:", completed_items["category"].isna().sum())

행 수: 0
category 결측: 0


In [102]:
print(orders["order_status"].unique())
print(orders["order_status"].value_counts())

<StringArray>
['배송중', '환불', '배송완료', '결제완료', '배송준비', '취소']
Length: 6, dtype: str
order_status
배송완료    4862
취소       458
환불       352
배송중      163
배송준비      92
결제완료      73
Name: count, dtype: int64


In [103]:
#52. 카테고리 합계 검증

category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

0
0
True


In [104]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']


In [105]:
print(len(completed_items))

0


In [106]:
completed_orders = orders[orders["order_status"] == "실제값"]
print("필터 후 행 수:", len(completed_orders))   # ← 이 한 줄

필터 후 행 수: 0


In [107]:
print(orders["order_status"].value_counts())

order_status
배송완료    4862
취소       458
환불       352
배송중      163
배송준비      92
결제완료      73
Name: count, dtype: int64


In [108]:
#50. 완료 주문상세와 상품 병합

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [109]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

0 0


product_match
left_only     0
right_only    0
both          0
Name: count, dtype: int64

In [110]:
#53. 상품별 매출


product_sales = (

    completed_items

    .groupby(

        ["product_id", "product_name", "category"],

        as_index=False,

    )

    .agg(

        total_sales=("line_total", "sum"),

        quantity_sold=("quantity", "sum"),

        order_count=("order_id", "nunique"),

        customer_count=("customer_id", "nunique"),

    )

    .sort_values("total_sales", ascending=False)

)

display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count


In [111]:
#46. 주문상세와 주문 병합

order_sales = (

    order_items_work

    .merge(

        orders_for_merge,

        on="order_id",

        how="left",

        validate="many_to_one",

        indicator="order_match",

    )

)

In [112]:
#47. 병합 검증

 

print("병합 전 행 수:", len(order_items_work))

print("병합 후 행 수:", len(order_sales))

display(

    order_sales["order_match"].value_counts(

        dropna=False

    )

)

병합 전 행 수: 14603
병합 후 행 수: 14603


order_match
both          14603
left_only         0
right_only        0
Name: count, dtype: int64

In [113]:
unmatched_orders = order_sales[

    order_sales["order_match"] != "both"

]

display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match


In [114]:
#48. 완료 주문 분석셋

 

display(

    order_sales["order_status"].value_counts(

        dropna=False

    )

)

order_status
배송완료    11764
취소       1128
환불        878
배송중       410
배송준비      232
결제완료      191
Name: count, dtype: int64

In [115]:
completed_sales = order_sales[

    order_sales["order_status"] == "completed"

].copy()

In [116]:
print("완료 주문상세 행:", len(completed_sales))

print(

    "완료 주문 수:",

    completed_sales["order_id"].nunique(),

)

print(

    "완료 주문 고객 수:",

    completed_sales["customer_id"].nunique(),

)

print(

    "완료 주문 매출:",

    completed_sales["line_total"].sum(),

)

완료 주문상세 행: 0
완료 주문 수: 0
완료 주문 고객 수: 0
완료 주문 매출: 0


In [117]:
#50. 완료 주문상세와 상품 병합

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [118]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

0 0


product_match
left_only     0
right_only    0
both          0
Name: count, dtype: int64

In [119]:
print(repr(orders["order_status"].unique()))

<StringArray>
['배송중', '환불', '배송완료', '결제완료', '배송준비', '취소']
Length: 6, dtype: str


In [120]:
# 1. 완료 주문 필터
completed_sales = orders[orders["order_status"] == "배송완료"]
print("완료 주문 수:", len(completed_sales))

# 2. 주문상세 + 상품 + 주문 붙이기
completed_items = (
    order_items
    .merge(products, on="product_id", how="inner")
    .merge(completed_sales, on="order_id", how="inner")
)
print("완료 주문상세 행 수:", len(completed_items))

# 3. 파생 컬럼
completed_items["line_total"] = (
    completed_items["quantity"] * completed_items["unit_price"]
)

# 4. 카테고리별 매출
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

완료 주문 수: 4862
완료 주문상세 행 수: 11764


,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
4,생활가전,377378400,1036,636,1814,1141
8,패션,251656100,1446,760,2601,1641
7,전자기기,227319900,1365,750,2473,1553
9,홈인테리어,139023300,866,578,1564,959
5,스포츠,128811400,920,605,1635,988
3,뷰티,108321000,1235,709,2357,1420
6,식품,91208800,1278,708,2357,1461
2,반려동물,86807600,787,533,1359,848
1,문구,35548200,741,524,1274,806
0,도서,35101200,872,579,1504,947


In [121]:
#51. 카테고리별 매출

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
.sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
4,생활가전,377378400,1036,636,1814,1141
8,패션,251656100,1446,760,2601,1641
7,전자기기,227319900,1365,750,2473,1553
9,홈인테리어,139023300,866,578,1564,959
5,스포츠,128811400,920,605,1635,988
3,뷰티,108321000,1235,709,2357,1420
6,식품,91208800,1278,708,2357,1461
2,반려동물,86807600,787,533,1359,848
1,문구,35548200,741,524,1274,806
0,도서,35101200,872,579,1504,947


In [122]:
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
4,생활가전,377378400,1036,636,1814,1141
8,패션,251656100,1446,760,2601,1641
7,전자기기,227319900,1365,750,2473,1553
9,홈인테리어,139023300,866,578,1564,959
5,스포츠,128811400,920,605,1635,988
3,뷰티,108321000,1235,709,2357,1420
6,식품,91208800,1278,708,2357,1461
2,반려동물,86807600,787,533,1359,848
1,문구,35548200,741,524,1274,806
0,도서,35101200,872,579,1504,947


In [123]:
#51. 카테고리별 매출

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
4,생활가전,377378400,1036,636,1814,1141
8,패션,251656100,1446,760,2601,1641
7,전자기기,227319900,1365,750,2473,1553
9,홈인테리어,139023300,866,578,1564,959
5,스포츠,128811400,920,605,1635,988
3,뷰티,108321000,1235,709,2357,1420
6,식품,91208800,1278,708,2357,1461
2,반려동물,86807600,787,533,1359,848
1,문구,35548200,741,524,1274,806
0,도서,35101200,872,579,1504,947


In [124]:
#52. 카테고리 합계 검증

category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

1481175900
1481175900
True


In [125]:
#53. 상품별 매출
product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
281,282,베이직 미니 가습기 베이지 P282,생활가전,25450200,73,42,42
51,52,컴팩트 공기청정기 베이지 P052,생활가전,24752000,72,44,44
191,192,플러스 미니 가습기 화이트 P192,생활가전,23005100,70,43,42
271,272,베이직 전기포트 화이트 P272,생활가전,21665800,69,48,47
291,292,컴팩트 미니 가습기 그린 P292,생활가전,19653900,71,39,39
171,172,프리미엄 미니 가습기 블루 P172,생활가전,18433600,60,36,35
290,291,베이직 보조배터리 블루 P291,전자기기,17980900,82,51,49
242,243,클래식 베이직 셔츠 그린 P243,패션,17759200,118,66,62
181,182,스마트 핸디 청소기 화이트 P182,생활가전,17234100,79,45,44
230,231,베이직 무선 마우스 그린 P231,전자기기,16467700,82,49,48


In [126]:
#54. 주문 날짜 변환과 주문 월 생성

completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [127]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [128]:
print(completed_items["order_date"].dtype)

datetime64[us]


In [129]:
completed_items["order_date"] = pd.to_datetime(completed_items["order_date"])

completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

print(completed_items["order_date"].dtype)
print(completed_items[["order_date", "order_month"]].head())

datetime64[us]
  order_date order_month
0 2025-05-08     2025-05
1 2025-05-08     2025-05
2 2025-06-20     2025-06
3 2025-06-20     2025-06
4 2025-05-31     2025-05


In [130]:
#54. 주문 날짜 변환과 주문 월 생성

completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [131]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [132]:
#55. 월별 매출

monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-01,70272900,244,197,903
1,2025-02,56964600,192,163,729
2,2025-03,71439800,260,207,971
3,2025-04,84479900,285,226,1091
4,2025-05,87100800,274,225,1089
5,2025-06,69060500,222,197,891
6,2025-07,73139100,251,218,953
7,2025-08,71355300,234,196,933
8,2025-09,79483000,244,208,996
9,2025-10,64780300,211,186,791


In [133]:
#56. 고객별 구매 금액

customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

In [134]:
#57. 고객 속성 연결

#개인정보 최소화를 위해 이름은 제외합니다.
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

In [135]:
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [136]:
display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

customer_match
both          1071
left_only        0
right_only       0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
882,989,5506900,10,51,여성,43,수원,both
426,477,5082300,11,53,여성,36,수원,both
675,754,5059900,10,37,남성,44,서울,both
689,772,4884800,13,52,남성,34,서울,both
911,1019,4869200,9,52,여성,40,인천,both
1014,1137,4829700,10,54,여성,28,부산,both
956,1070,4618600,8,47,여성,27,수원,both
215,243,4567000,9,41,여성,48,인천,both
808,906,4551300,12,43,남성,25,서울,both
1,2,4537900,8,47,여성,27,고양,both


In [137]:
#58. 결과 폴더 생성
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

c:\dev\2team_shopingmall_dash_board\reports\chapter04


In [138]:
#59. 결과 파일 저장
outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 469
product_sales.csv True 20831
monthly_sales.csv True 648
customer_sales.csv True 42276


In [139]:
#60. 저장 결과 다시 읽기
saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,생활가전,377378400,1036,636,1814,1141
1,패션,251656100,1446,760,2601,1641
2,전자기기,227319900,1365,750,2473,1553
3,홈인테리어,139023300,866,578,1564,959
4,스포츠,128811400,920,605,1635,988


(10, 6)


In [140]:
#61. 병합 점검 함수
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

In [141]:
#위에 만든 함수 호출
check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)


[주문상세-주문]
병합 전 행 수: 14603
병합 후 행 수: 14603
order_match
both          14603
left_only         0
right_only        0
Name: count, dtype: int64


In [142]:
#62. 집계 합계 검증 함수
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [143]:
check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

[카테고리별 매출]
원본 합계: 1481175900
요약 합계: 1481175900
차이: 0


## 64.pandas 코드 요청 프롬프트

나는 온라인 쇼핑몰 데이터를 pandas로 분석하고 있습니다.

분석 목표:
완료 주문 기준 카테고리별 매출을 계산합니다.

DataFrame과 한 행의 의미:
- orders: 주문 한 건
- order_items: 주문에 포함된 상품 한 항목
- products: 상품 한 개

실제 컬럼:
- orders:
  order_id, customer_id, order_date,
  payment_method, order_status
- order_items:
  order_item_id, order_id, product_id,
  quantity, unit_price
- products:
  product_id, product_name, category, price

주요 관계:
- order_items.order_id → orders.order_id
  many_to_one
- order_items.product_id → products.product_id
  many_to_one

분석 범위:
- order_status가 배송완료인 주문만 포함
- line_total = quantity × unit_price
- 주문 수는 order_id의 고유 개수
- 매출은 line_total 합계

원하는 결과:
- category
- total_sales
- order_count
- customer_count
- quantity_sold

검증 요구사항:
1. 각 merge에 validate를 사용해 주세요.
2. indicator로 미매칭을 확인해 주세요.
3. 병합 전후 행 수를 출력해 주세요.
4. 카테고리 합계와 완료 주문 전체 합계를 비교해 주세요.
5. 실제로 존재하지 않는 컬럼을 만들지 마세요.
6. 코드 실행 전 확인할 항목도 설명해 주세요.

## 65. LLM 코드 검증표

| 검증 항목 | 확인 내용 | 결과 | 근거 |
| --- | --- | --- | --- |
| DataFrame | 실제 변수명과 같은가? | ✅ 통과 | orders / order_items / products / customers 일치 |
| 컬럼 | 실제 컬럼만 사용하는가? | ✅ 통과 | 존재하지 않는 컬럼 생성 없음 |
| 상태값 | `completed` 표기가 맞는가? | ❌ 실패 | 실제값은 `배송완료`. 필터 결과 0행 |
| 계산식 | `quantity × unit_price`인가? | ✅ 통과 | `products.price` 미사용, 수작업 검증 일치 |
| 분석 범위 | 완료 주문만 포함하는가? | ⚠️ 조건부 | 중간 경로 무효, 최종 경로만 정상 |
| 주문 수 | `nunique()`를 사용하는가? | ✅ 통과 | `order_count=("order_id", "nunique")` |
| 병합 키 | 실제 관계와 맞는가? | ✅ 통과 | order_id · product_id 모두 both 14,603 |
| validate | `many_to_one`이 적용되었는가? | ❌ 실패 | 최종 merge에 validate 인자 없음 |
| indicator | 미매칭을 확인하는가? | ❌ 실패 | 최종 merge가 `how="inner"`, indicator 없음 |
| 행 수 | 병합 전후를 비교하는가? | ⚠️ 조건부 | 무효 경로만 비교, 최종 경로 미검증 |
| 합계 | 원본과 요약 합계를 비교하는가? | ✅ 통과 | 1,481,175,900 = 1,481,175,900 |
| 개인정보 | 원본 고객 정보를 요구하지 않는가? | ⚠️ 주의 | 집계는 name 제외, 셀 출력에 실명 잔존 |